In [ ]:
### Applies Friedman’s test to check for overall significant differences. Friedman + Nemenyi
# Select only the columns of interest
metric = "hypervolume_norm"  # hypervolume, hypervolume_norm, generations, time, spacing_norm
selected_columns = ["directory", metric]
subset_df = metrics_df[selected_columns]

# Group by directory
grouped_df = subset_df.groupby("directory")

# Ensure all algorithms have the same number of executions
hv_values = [group[metric].values for _, group in grouped_df]
min_length = min(len(hv) for hv in hv_values)
hv_values = [hv[:min_length] for hv in hv_values]

# Friedman test
friedman_stat, p_value = friedmanchisquare(*hv_values)
print(f"Friedman test statistic: {friedman_stat}, p-value: {p_value}")

if p_value < 0.05:
    print("Significant differences found. Applying Nemenyi post-hoc test...")

    # Transform data into matrix format for posthoc analysis
    hv_matrix = np.array(hv_values).T  # Transpose so each column represents an algorithm

    # Apply Nemenyi test
    nemenyi_results = sp.posthoc_nemenyi_friedman(hv_matrix)

    # Convert results to DataFrame for better visualization
    directories = list(grouped_df.groups.keys())  # Ensure directories is a list of strings
    nemenyi_results.columns = directories
    nemenyi_results.index = directories
    print("Nemenyi post-hoc test results (p-values):")
    print(nemenyi_results)

    # Identify significantly different algorithms
    print("\nIdentifying significantly different algorithms...")
    alpha = 0.05  # Significance level
    significant_pairs = []

    # Iterate over the matrix to find significant pairs
    for i in range(len(directories)):
        for j in range(i + 1, len(directories)):
            if nemenyi_results.iloc[i, j] < alpha:
                significant_pairs.append((directories[i], directories[j], nemenyi_results.iloc[i, j]))

    # Print results
    if significant_pairs:
        print("\nSignificantly different algorithms:")
        for pair in significant_pairs:
            print(f"- {pair[0]} and {pair[1]} (p-value: {pair[2]:.4f})")
    else:
        print("\nNo significantly different algorithms found.")
else:
    print("No significant differences found between algorithms.")
 
 
 
# Boxplot for hypervolume by directory
plt.figure(figsize=(12, 6)) # Set the figure size
metrics_df.boxplot(column=["generations"], by="directory")
plt.title("Boxplot of Generations by Directory")
plt.suptitle("") # Remove the default super title
plt.xlabel("Directory") # Label for the x-axis
plt.ylabel("Hypervolume") # Label for the y-axis
plt.xticks(rotation=45) # Rotate x-axis labels for better readability
plt.show()


In [ ]:
### Applies Friedman’s test to check for overall significant differences. Friedman + Nemenyi
# Select only the columns of interest
metric = "hypervolume_norm"  # hypervolume, hypervolume_norm, generations, time, spacing_norm
selected_columns = ["directory", metric]
subset_df = metrics_df[selected_columns]

# Group by directory
grouped_df = subset_df.groupby("directory")

# Ensure all algorithms have the same number of executions
hv_values = [group[metric].values for _, group in grouped_df]
min_length = min(len(hv) for hv in hv_values)
hv_values = [hv[:min_length] for hv in hv_values]

# Friedman test
friedman_stat, p_value = friedmanchisquare(*hv_values)
print(f"Friedman test statistic: {friedman_stat}, p-value: {p_value}")

if p_value < 0.05:
    print("Significant differences found. Applying Nemenyi post-hoc test...")

    # Transform data into matrix format for posthoc analysis
    hv_matrix = np.array(hv_values).T  # Transpose so each column represents an algorithm

    # Apply Nemenyi test
    nemenyi_results = sp.posthoc_nemenyi_friedman(hv_matrix)

    # Convert results to DataFrame for better visualization
    directories = list(grouped_df.groups.keys())  # Ensure directories is a list of strings
    nemenyi_results.columns = directories
    nemenyi_results.index = directories
    print("Nemenyi post-hoc test results (p-values):")
    print(nemenyi_results)

    # Identify significantly different algorithms
    print("\nIdentifying significantly different algorithms...")
    alpha = 0.05  # Significance level
    significant_pairs = []

    # Iterate over the matrix to find significant pairs
    for i in range(len(directories)):
        for j in range(i + 1, len(directories)):
            if nemenyi_results.iloc[i, j] < alpha:
                significant_pairs.append((directories[i], directories[j], nemenyi_results.iloc[i, j]))

    # Print results
    if significant_pairs:
        print("\nSignificantly different algorithms:")
        for pair in significant_pairs:
            print(f"- {pair[0]} and {pair[1]} (p-value: {pair[2]:.4f})")
    else:
        print("\nNo significantly different algorithms found.")
else:
    print("No significant differences found between algorithms.")
 
 
 
# Boxplot for hypervolume by directory
plt.figure(figsize=(12, 6)) # Set the figure size
metrics_df.boxplot(column=["generations"], by="directory")
plt.title("Boxplot of Generations by Directory")
plt.suptitle("") # Remove the default super title
plt.xlabel("Directory") # Label for the x-axis
plt.ylabel("Hypervolume") # Label for the y-axis
plt.xticks(rotation=45) # Rotate x-axis labels for better readability
plt.show()


In [2]:
import pandas as pd
from pathlib import Path
import numpy as np

# Ruta base
base_dir = Path("results")

alg_dirs = ["MOEAD", "NSGA2", "SMSEMOA"]
csv_paths = [base_dir / alg / "runs_summary.csv" for alg in alg_dirs]

# ✅ Columnas objetivo a normalizar (ajusta si quieres usar *_mean_feas)
f1_col = "f1_min_feas"
f2_col = "f2_min_feas"

# 1️⃣ Leer todos los CSV
dfs = []
for path in csv_paths:
    df = pd.read_csv(path)
    df["_source_csv"] = path
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# 2️⃣ Min y max globales para f1 y f2
global_f1_min = all_df[f1_col].min()
global_f1_max = all_df[f1_col].max()
global_f2_min = all_df[f2_col].min()
global_f2_max = all_df[f2_col].max()

print(f"{f1_col} global min = {global_f1_min:.6f}")
print(f"{f1_col} global max = {global_f1_max:.6f}")
print(f"{f2_col} global min = {global_f2_min:.6f}")
print(f"{f2_col} global max = {global_f2_max:.6f}")

# 3️⃣ Normalización Min-Max (robusta si max == min)
f1_denom = global_f1_max - global_f1_min
f2_denom = global_f2_max - global_f2_min

all_df["f1_norm"] = np.where(
    f1_denom == 0,
    0.0,
    (all_df[f1_col] - global_f1_min) / f1_denom
)

all_df["f2_norm"] = np.where(
    f2_denom == 0,
    0.0,
    (all_df[f2_col] - global_f2_min) / f2_denom
)

# (Opcional) forzar rango [0, 1] por estabilidad numérica
all_df["f1_norm"] = all_df["f1_norm"].clip(0, 1)
all_df["f2_norm"] = all_df["f2_norm"].clip(0, 1)

# 4️⃣ Guardar cada CSV con las nuevas columnas
for path in csv_paths:
    df_alg = all_df[all_df["_source_csv"] == path].drop(columns="_source_csv")
    df_alg.to_csv(path, index=False)
    print(f"✔ Guardado: {path}")


f1_min_feas global min = -0.982929
f1_min_feas global max = -0.638360
f2_min_feas global min = 1.000000
f2_min_feas global max = 6.000000
✔ Guardado: results/MOEAD/runs_summary.csv
✔ Guardado: results/NSGA2/runs_summary.csv
✔ Guardado: results/SMSEMOA/runs_summary.csv


In [4]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp
import matplotlib.pyplot as plt
from pathlib import Path

base = Path("results")
alg_dirs = ["MOEAD", "NSGA2", "SMSEMOA"]

dfs = []
for d in alg_dirs:
    df = pd.read_csv(base / d / "runs_summary.csv")
    dfs.append(df)

metrics_df = pd.concat(dfs, ignore_index=True)


In [5]:
metric = "hypervolume_norm"

pivot = metrics_df.pivot_table(
    index="run_id",          # bloque
    columns="algorithm",     # tratamientos
    values=metric
)
pivot


algorithm,NSGA2,moead,smsemoa
run_id,,,
1,0.953093,0.402397,0.944197
2,0.931133,0.826179,0.962995
3,0.971334,0.000000,0.946606
4,0.952554,0.703541,0.928544
5,1.000000,0.778453,0.937674
6,0.973311,0.581500,0.967914
7,0.979024,0.784162,0.962816
8,0.961182,0.674672,0.964183
9,0.959071,0.755097,0.970699


In [6]:
friedman_stat, p_value = friedmanchisquare(
    *[pivot[col].values for col in pivot.columns]
)

print(f"Friedman χ² = {friedman_stat:.4f}, p-value = {p_value:.6f}")


Friedman χ² = 15.2000, p-value = 0.000500


In [7]:
import scikit_posthocs as sp
import numpy as np

# Matriz en formato correcto (filas = bloques, columnas = algoritmos)
data_matrix = pivot.values  

nemenyi = sp.posthoc_nemenyi_friedman(data_matrix)

nemenyi.columns = pivot.columns
nemenyi.index = pivot.columns

print(nemenyi)


algorithm     NSGA2     moead   smsemoa
algorithm                              
NSGA2      1.000000  0.001010  0.895638
moead      0.001010  1.000000  0.004965
smsemoa    0.895638  0.004965  1.000000


In [8]:
ranks = pivot.rank(axis=1, ascending=False)
print(ranks.mean())


algorithm
NSGA2      1.4
moead      3.0
smsemoa    1.6
dtype: float64
